# Train cybersec_agent on a cloud GPU (Google Colab)

Runs the same pipeline as the laptop setup: QLoRA training with `train.py`,
then the held-out evaluation. Your laptop isn't needed.

**Before you start**

1. **Runtime → Change runtime type → T4 GPU** (free tier), then Save.
2. The repo is private, so Colab needs a GitHub token to clone it. Create a
   fine-grained token at github.com/settings/tokens with read-only *Contents*
   access to `cybersec_agent`, then add it in Colab's **Secrets** panel (key
   icon on the left) as `GITHUB_TOKEN` and turn on notebook access. Skip this
   if the repo is public.
3. Optional, for scoring: add your Anthropic API key as the secret
   `ANTHROPIC_API_KEY`.

Then **Runtime → Run all**. Colab's free tier can disconnect after long idle
periods, so keep the tab open. Everything in `/content` is deleted when the
session ends; the last cell downloads the results.

In [ ]:
!nvidia-smi

## 1. Get the code

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/Amir24583/cybersec_agent.git"
except Exception:
    url = "https://github.com/Amir24583/cybersec_agent.git"   # public repo

if not os.path.exists("/content/cybersec_agent"):
    !git clone --depth 1 {url} /content/cybersec_agent
%cd /content/cybersec_agent
!git log --oneline -1

## 2. Install

Colab already has a CUDA build of PyTorch that supports its GPUs, so the
laptop's cu128 and `--no-deps` steps aren't needed here.

In [ ]:
!pip install -q unsloth anthropic

## 3. Check the eval set doesn't overlap the training data

In [ ]:
!python scripts/decontam.py

## 4. Train

QLoRA on `data/seed_scaled.jsonl` (928 rows). The adapter is saved to
`outputs/cybersec-lora`. Watch the loss go down in the log.

In [ ]:
!python train.py

## 5. Answer the eval questions with both models

In [ ]:
!python eval_generate.py --model base
!python eval_generate.py --model finetuned

## 6. Score the answers (optional, needs `ANTHROPIC_API_KEY`)

Costs about $2. If you skip it here, run `python scripts/judge.py` on your
laptop after copying `outputs/eval/` into the repo; it doesn't need a GPU.

In [ ]:
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    !python scripts/judge.py
except Exception as e:
    print("Skipping scoring:", e)

## 7. Retrieval (RAG): look facts up instead of training them in

The 3B fine-tune learned the answer style but not the facts (30% vs 29% key
points, 1.75 vs 2.00 accuracy). This section gives the model reference passages
at answer time instead: MITRE ATT&CK, the CISA KEV catalog, the OWASP cheat
sheets, Wikipedia security articles and the training answers, searched with
BM25 (`rag.py`). See `docs/rag.md`.

It needs no GPU training and no adapter: the main comparison is the **base**
model with and without retrieval, on the same 40 held-out questions. You can
start a fresh runtime here and run sections 1, 2 and then this one.

In [ ]:
!python scripts/build_corpus.py
!python scripts/retrieval_check.py --k 4 | tail -7

Answer the 40 questions with retrieval. The 3B base takes about 15 minutes on a
T4; the 7B base about 30. The fine-tuned run only works if `outputs/cybersec-lora`
exists (from section 4 in this runtime).

In [ ]:
!python eval_generate.py --model base                 # no-RAG reference (skip if already run)
!python eval_generate.py --model base --rag
!python eval_generate.py --model base --rag --base-model unsloth/Qwen2.5-7B-Instruct --name base7b_rag
if os.path.exists("outputs/cybersec-lora"):
    !python eval_generate.py --model finetuned --rag

Score each RAG run against the plain 3B base (optional, needs
`ANTHROPIC_API_KEY`). Without a key, download the zip in section 8 and the
answers can be graded the same blind way as the first eval.

In [ ]:
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    !python scripts/judge.py --ref base --cand base_rag
    !python scripts/judge.py --ref base --cand base7b_rag
except Exception as e:
    print("Skipping scoring:", e)

## 8. Download the results

`results.zip` has the LoRA adapter (`outputs/cybersec-lora`) and the eval
answers and scores (`outputs/eval`). Unzip it into the repo on your laptop to
use `chat.py` there.

In [ ]:
!rm -rf outputs/cybersec-lora/checkpoint-*
!zip -qr /content/results.zip outputs
from google.colab import files
files.download("/content/results.zip")